# Program-only chunk quality audit

Notebook 10 completed the nonparagraph review replay. This notebook applies every whole-document input hold, reconciles the exact passage holds, and freezes a separate probability sample of parent documents whose offsets come from the program alone. All offsets are half-open Unicode code point ranges on the bound Notebook 05 body.

The offset-only candidate uses a 512-content-token `cl100k_base` reference. Adoption requires the selected model's tokenizer, special-token budget and a replay of the exact gate if offsets change. Legacy receipts remain immutable. This stage does not create labels or fit models.

## Bind completed review, reference chunks and legacy decisions

Read the closed Notebook 10 replay and the earlier exact-hold ledger.

In [ ]:
import json
import math
import tempfile
from collections import Counter
from pathlib import Path

import nbformat
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import Window
from pyspark.sql import functions as F

ROOT = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks/10_full_nonparagraph_review.ipynb").is_file()
)
NOTEBOOK = ROOT / "notebooks/11_program_chunk_qa.ipynb"
SOURCE = ROOT / "notebooks/10_full_nonparagraph_review.ipynb"
n10 = nbformat.read(SOURCE, as_version=4)
sources = {cell.id: cell.source for cell in n10.cells if cell.cell_type == "code"}
upstream = {"__name__": "__main__"}

# Only reuse the existing receipt-validation branches of Notebook 10.
run = ROOT / ".cache/chunk-policy-review/full-nonparagraph-v1"
closure_path = run / "FULL_REVIEW_CLOSURE.json"
exact_path = run / "FULL_CANDIDATE_EXACT_RECHECK.json"
closure = json.loads(closure_path.read_text(encoding="utf-8"))
prior_exact_audit = json.loads(exact_path.read_text(encoding="utf-8"))
if closure["status"] != "full_nonparagraph_review_and_legacy_adjudication_complete":
    raise ValueError("The full review receipt does not record completed replay.")
for name in ("10-bind", "10-packets"):
    exec(compile(sources[name], f"{SOURCE.name}:{name}", "exec"), upstream)
sha256_file = upstream["sha256_file"]
canonical_sha256 = upstream["canonical_sha256"]
for label, assignment in upstream["assignments"].items():
    replay = run / "review/replays" / f"{label}.json"
    receipt = run / "review" / assignment["receipt"]
    if not replay.is_file() or not receipt.is_file():
        raise ValueError(f"The completed review is missing a receipt or replay for {label}.")
name = "10-replay"
exec(compile(sources[name], f"{SOURCE.name}:{name}", "exec"), upstream)

case_path = run / "PRIOR_CASE_REPLAY.json"
case_receipt = json.loads(case_path.read_text(encoding="utf-8"))
case_sha = sha256_file(case_path)
if (
    sha256_file(upstream["manifest_path"]) != closure["review_manifest_sha256"]
    or case_sha != closure["legacy_case_replay_sha256"]
    or case_receipt["case_manifest_sha256"] != sha256_file(upstream["case_manifest_path"])
    or case_receipt["sample_closure_sha256"] != upstream["sample_closure_sha"]
    or case_receipt["remaining_escalated"]
    or case_receipt["remaining_reference_budget"]
    or case_receipt["validation_failures"]
    or prior_exact_audit["review_closure_sha256"] != sha256_file(closure_path)
):
    raise ValueError("The closure, legacy decisions and exact hold gate failed binding.")
case_results = {row["release_row"]: row for row in case_receipt["decisions"]}
reviewed = upstream["reviewed"]
prior = upstream["prior"]
if (
    len(case_receipt["decisions"]) != len(case_results)
    or set(case_results) != upstream["pending"]
    or set(reviewed) != upstream["parent_roster"]
):
    raise ValueError("The verified disposition rosters do not match the closed review scope.")

old_gate = run / "exact_candidate_gate_v1"
old_manifest_path = old_gate / "MANIFEST.json"
old_manifest = json.loads(old_manifest_path.read_text(encoding="utf-8"))
if old_manifest["audit"] != prior_exact_audit:
    raise ValueError("The previous exact hold ledger does not match its receipt.")
old_holds = upstream["load_table"](old_gate, "held_occurrences", old_manifest["files"])
KEY = ["release_row", "start", "end", "chunk_sha256"]
old_keys = old_holds.select(*KEY)
if old_keys.distinct().count() != prior_exact_audit["held_occurrences"]:
    raise ValueError("The previous exact hold keys are missing or duplicated.")

spark = upstream["spark"]
plans = upstream["plans"]
parents = upstream["parents"]
budget = upstream["budget"]
encoding = upstream["encoding"]
former_control = upstream["sample_closure"]["stratified_summary"]["paragraph_control"]
identity = {
    "notebook_sha256": sha256_file(NOTEBOOK),
    "review_notebook_sha256": sha256_file(SOURCE),
    "reference_manifest_sha256": upstream["policy"]["reference_manifest_sha256"],
    "full_review_closure_sha256": sha256_file(closure_path),
    "prior_case_replay_sha256": case_sha,
    "previous_exact_hold_manifest_sha256": sha256_file(old_manifest_path),
}


## Reconstruct parent dispositions

Keep pending semantic cases, pending reference-budget cases and confirmed exclusions separately identifiable. Apply verified repairs and preserve the previous paragraph-control decisions. The full nonparagraph review coverage count measures replay; input eligibility follows the disposition.

In [ ]:
complete = "boundary_review_complete"
explicit_hold = "hold_for_model_input"
dispositions = []
overrides = []
for key, row in {**prior, **reviewed}.items():
    status = row["status"]
    ends = row["internal_end_offsets"]
    counts = row["token_counts"]
    case = case_results.get(key)
    if case is not None:
        action = case["disposition"]
        status = explicit_hold if action == explicit_hold else complete
        ends = case["replacement_ends"][:-1] if action == "repair_reference" else None
        counts = case["reference_token_counts"]
    if status == complete and ends is not None:
        if len(counts) != len(ends) + 1:
            raise ValueError("Reviewed ends do not reconcile to their reference token counts.")
        overrides.append((key, row["body_sha256"], ends, counts))
    dispositions.append(
        (
            key, row["body_sha256"], row["ml_split"], row["sample"], status,
            row.get("review_origin"), row.get("adjudication_receipt_sha256"),
            case["receipt_sha256"] if case is not None else None,
        )
    )

review_ledger = spark.createDataFrame(
    dispositions,
    "release_row long, clean_sha256 string, ml_split string, review_scope string, "
    "disposition string, review_origin string, sample_adjudication_receipt_sha256 string, "
    "legacy_adjudication_receipt_sha256 string",
)
nonparagraph_status = Counter(row[4] for row in dispositions if row[3] == "nonparagraph")
if dict(nonparagraph_status) != closure["current_nonparagraph_dispositions"]:
    raise ValueError("Reconstructed dispositions differ from the closed review receipt.")
if len(overrides) != prior_exact_audit["reviewed_parent_plans_used"]:
    raise ValueError("Reviewed overrides differ from the previous candidate.")

plan_keys = plans.select("release_row", "clean_sha256", "ml_split")
if review_ledger.join(plan_keys, plan_keys.columns, "left_anti").count():
    raise ValueError("A reviewed record disagrees with its formal body or inherited split.")
parent_holds = review_ledger.where(F.col("disposition") != complete)
parent_holds_total = parent_holds.count()
if parent_holds_total != sum(
    count for status, count in nonparagraph_status.items() if status != complete
):
    raise ValueError("A paragraph-control disposition also requires an input hold.")
reviewed_offsets = spark.createDataFrame(
    overrides,
    "release_row long, clean_sha256 string, reviewed_ends array<int>, reviewed_counts array<int>",
)


## Check geometry and rebuild the exact occurrence gate

Apply whole-parent holds before grouping active chunks by exact slice hash and length. Carry forward still-matching previous exact occurrence holds by parent, range and slice hash. Apply the union with current cross-split exact matches. Chunk positions and original ordinals remain available when an exact hold creates a gap. Semantic near-duplicate screening remains covered by upstream evidence and its documented limitations.

In [ ]:
# Apply whole-parent exclusions before deriving chunk occurrences or leakage groups.
held_ids = parent_holds.select("release_row")
excluded_reference_chunks = (
    plans.join(F.broadcast(held_ids), "release_row")
    .agg(F.sum(F.size("plan.ends"))).first()[0]
)
eligible_plans = (
    plans.join(F.broadcast(held_ids), "release_row", "left_anti")
    .join(F.broadcast(reviewed_offsets), ["release_row", "clean_sha256"], "left")
    .join(F.broadcast(review_ledger.select("release_row", "review_scope")), "release_row", "left")
    .withColumn(
        "ends",
        F.when(
            F.col("reviewed_ends").isNotNull(),
            F.concat("reviewed_ends", F.array("clean_chars")),
        ).otherwise(F.col("plan.ends")),
    )
    .withColumn("counts", F.coalesce("reviewed_counts", "plan.counts"))
    .withColumn(
        "plan_origin",
        F.when(F.col("reviewed_ends").isNotNull(), "reviewed_offsets")
        .when(F.col("review_scope").isNotNull(), "reviewed_reference")
        .otherwise("program_only"),
    )
    .select(
        "release_row", "native_identity", "clean_sha256", "clean_chars", "ml_split", "format",
        "ends", "counts", "plan_origin", F.col("plan.boundary_kinds").alias("reference_kinds"),
    )
    .persist(StorageLevel.DISK_ONLY)
)
program_plans = eligible_plans.where(F.col("plan_origin") == "program_only")
geometry = eligible_plans.select(
    "release_row", "clean_sha256", "ml_split", "ends", "counts", "clean_chars",
    "plan_origin", "reference_kinds", F.posexplode("ends").alias("ordinal", "end"),
).withColumn("start", F.coalesce(F.get("ends", F.col("ordinal") - 1), F.lit(0)))
geometry = geometry.withColumn(
    "reference_token_count", F.get("counts", "ordinal")
).withColumn("codepoints", F.col("end") - F.col("start"))
body_index = parents.select(
    "release_row", "clean_sha256", "clean_text",
    F.transform("paragraphs", lambda span: span.start).alias("paragraph_starts"),
)
validated = geometry.join(body_index, ["release_row", "clean_sha256"])
valid_span = (
    (F.col("start") >= 0)
    & (F.col("codepoints") > 0)
    & (F.size("ends") == F.size("counts"))
    & (F.try_element_at("ends", F.lit(-1)) == F.col("clean_chars"))
    & (F.col("end") <= F.length("clean_text"))
    & F.col("reference_token_count").between(1, budget)
)
program_edge = (F.col("end") == F.col("clean_chars")) | (
    (F.get("reference_kinds", "ordinal") == "paragraph")
    & F.array_contains("paragraph_starts", F.col("end"))
)
candidate_columns = [
    "release_row", "clean_sha256", "ml_split", "ordinal", "start", "end", "codepoints",
    "reference_token_count", "plan_origin",
]
chunks = (
    validated.select(
        *candidate_columns,
        F.sha2(F.substring("clean_text", F.col("start") + 1, "codepoints"), 256).alias(
            "chunk_sha256"
        ),
        F.coalesce(
            valid_span & ((F.col("plan_origin") != "program_only") | program_edge), F.lit(False)
        ).alias("valid"),
    )
    .persist(StorageLevel.DISK_ONLY)
)
stats = chunks.agg(
    F.count("*").alias("chunks"),
    F.count_distinct("release_row").alias("parents"),
    F.count_if(~F.col("valid")).alias("invalid_chunks"),
    F.max("reference_token_count").alias("max_reference_tokens"),
).first().asDict()
expected = prior_exact_audit["candidate_chunks"] - excluded_reference_chunks
if (
    stats["chunks"] != expected
    or stats["parents"] != upstream["upstream08"]["reference_audit"]["parents"] - parent_holds_total
    or stats["invalid_chunks"]
):
    raise ValueError(f"Candidate geometry or parent coverage failed: {stats}")

def cross_split_groups(frame):
    return frame.groupBy("chunk_sha256", "codepoints").agg(
        F.min("ml_split").alias("first_split"), F.max("ml_split").alias("last_split")
    ).where(F.col("first_split") != F.col("last_split")).select("chunk_sha256", "codepoints")


candidate = chunks.drop("valid")
groups = cross_split_groups(candidate).persist(StorageLevel.DISK_ONLY)
retained = candidate.join(F.broadcast(old_keys), KEY).select(
    *KEY, F.lit(True).alias("prior_exact_hold")
)
detected = candidate.join(groups, ["chunk_sha256", "codepoints"]).select(
    *KEY, F.lit(True).alias("current_exact_cross_split")
)
exact_holds = (
    retained.join(detected, KEY, "full")
    .na.fill(False, subset=["prior_exact_hold", "current_exact_cross_split"])
    .persist(StorageLevel.DISK_ONLY)
)
model_candidate = (
    candidate.join(exact_holds.select(*KEY), KEY, "left_anti").persist(StorageLevel.DISK_ONLY)
)
remaining_exact = cross_split_groups(model_candidate).count()
remaining_parent_holds = model_candidate.join(F.broadcast(held_ids), "release_row").count()
hold_stats = exact_holds.agg(
    F.count("*").alias("held"),
    F.count_if("prior_exact_hold").alias("retained"),
    F.count_if(~F.col("prior_exact_hold")).alias("added"),
).first().asDict()
input_stats = model_candidate.agg(
    F.count("*").alias("chunks"), F.count_distinct("release_row").alias("parents")
).first().asDict()
if (
    remaining_exact
    or remaining_parent_holds
    or input_stats["chunks"] + hold_stats["held"] != stats["chunks"]
):
    raise ValueError("The parent and exact occurrence gates did not reconcile.")
summary = {
    "replay_status": "complete",
    "disposition_status": "held_parents_excluded",
    "nonparagraph_dispositions": dict(nonparagraph_status),
    "nonparagraph_open_cases": sum(
        count for status, count in nonparagraph_status.items()
        if status not in {explicit_hold, complete}
    ),
    "nonparagraph_material_first_pass": closure["material_nonparagraph_parents_first_pass"],
    "whole_parent_holds": parent_holds_total,
    "chunks_excluded_by_whole_parent_holds": excluded_reference_chunks,
    "chunks_after_parent_gate": stats["chunks"],
    "exact_cross_split_groups_after_parent_gate": groups.count(),
    "exact_occurrences_held": hold_stats["held"],
    "previous_exact_occurrences_retained": hold_stats["retained"],
    "new_exact_occurrences_held": hold_stats["added"],
    "remaining_exact_cross_split_groups": remaining_exact,
    "remaining_held_parent_chunks": remaining_parent_holds,
    "reference_candidate_chunks": input_stats["chunks"],
    "reference_candidate_parents": input_stats["parents"],
    "parents_without_chunks_after_exact_holds": stats["parents"] - input_stats["parents"],
    "invalid_candidate_chunks": stats["invalid_chunks"],
    "reference_encoding": encoding,
    "reference_content_token_budget": budget,
    "max_reference_content_tokens": stats["max_reference_tokens"],
    "actual_model_tokenizer_status": "pending_model_selection",
}
display(pd.DataFrame(summary.items(), columns=["Measure", "Value"]))


## Draw a fresh program-only sample

The program-only frame uses the already materialized paragraph-preferred reference plan from Notebook 06. Every internal edge in this frame must be a recorded paragraph start. The reference planner chooses boundaries within a token estimate and checks each resulting slice. This audit keeps its offsets fixed.

Draw 1,150 parents with internal paragraph cuts and 350 single-chunk parents. Allocate within boundary kind, split, format and character length band, with at least one draw per nonempty stratum. Record the frozen hash order, stratum sizes and inverse inclusion weights. The earlier 500 controls were already reviewed and form a separate cohort. Their first-pass findings are retained as historical evidence.

In [ ]:
sample_seed = "program-chunk-qa-v1-20261122"
targets = {"paragraph_only": 1150, "terminal_only": 350}
strata = ["boundary_kind", "ml_split", "format", "length_band"]
population = (
    program_plans.withColumn(
        "boundary_kind", F.when(F.size("ends") > 1, "paragraph_only").otherwise("terminal_only")
    )
    .withColumn(
        "length_band",
        F.when(F.col("clean_chars") <= 1200, "up_to_1200")
        .when(F.col("clean_chars") <= 4800, "1201_to_4800")
        .when(F.col("clean_chars") <= 9600, "4801_to_9600")
        .otherwise("over_9600"),
    )
    .withColumn("format", F.coalesce("format", F.lit("Unspecified")))
)
counts = population.groupBy(*strata).count().orderBy(*strata).collect()
allocations = []
for kind, target in targets.items():
    part = [row for row in counts if row.boundary_kind == kind]
    available = sum(row["count"] for row in part)
    if not len(part) <= target <= available:
        raise ValueError(f"Cannot assign at least one QA parent to every stratum for {kind}.")
    # One parent per nonempty stratum, plus proportional allocation of remaining capacity.
    seats = target - len(part)
    capacity = available - len(part)
    shares = [
        divmod(seats * (row["count"] - 1), capacity) if capacity else (0, 0) for row in part
    ]
    quotas = [1 + quotient for quotient, _ in shares]
    order = sorted(range(len(part)), key=lambda i: (-shares[i][1], i))
    for i in order[:target - sum(quotas)]:
        quotas[i] += 1
    for row, quota in zip(part, quotas, strict=True):
        if not 1 <= quota <= row["count"]:
            raise ValueError("A QA stratum has an invalid quota.")
        allocations.append((*[row[key] for key in strata], row["count"], quota))

sample_strata = spark.createDataFrame(allocations, [*strata, "population", "sample_quota"])
selection = (
    population.join(F.broadcast(sample_strata), strata)
    .withColumn(
        "draw_key",
        F.sha2(F.concat_ws("|", F.lit(sample_seed), "native_identity", "release_row"), 256),
    )
    .withColumn(
        "stratum_rank",
        F.row_number().over(Window.partitionBy(*strata).orderBy("draw_key", "release_row")),
    )
    .where(F.col("stratum_rank") <= F.col("sample_quota"))
    .withColumn("sampling_weight", F.col("population") / F.col("sample_quota"))
    .drop("native_identity", "counts", "reference_kinds")
    .persist(StorageLevel.DISK_ONLY)
)
selection_counts = {
    row.boundary_kind: row["count"] for row in selection.groupBy("boundary_kind").count().collect()
}
if selection_counts != targets or selection.join(
    F.broadcast(review_ledger.select("release_row")), "release_row"
).count():
    raise ValueError("The fresh QA draw has an unexpected size or repeats earlier review.")
actual_by_stratum = selection.groupBy(*strata).count()
if sample_strata.join(actual_by_stratum, strata, "left").where(
    F.col("count").isNull() | (F.col("count") != F.col("sample_quota"))
).count():
    raise ValueError("The actual selection differs from the frozen stratum quotas.")

population_counts = {
    kind: sum(row["count"] for row in counts if row.boundary_kind == kind) for kind in targets
}
whole_reference_counts = {
    "paragraph_only": 246514,
    "terminal_only": 455276,
}
excluded_prior_controls = sum(
    whole_reference_counts[kind] - population_counts[kind] for kind in targets
)
if excluded_prior_controls != former_control["parents"]:
    raise ValueError("The unsupervised program scope and prior control cohort do not reconcile.")
summary.update({
    "program_only_frame": population_counts,
    "program_structure_reference_scope": whole_reference_counts,
    "reviewed_controls_excluded_from_fresh_draw": excluded_prior_controls,
    "prior_control_first_pass": former_control,
    "fresh_sample_parents": selection_counts,
    "fresh_sample_strata": len(allocations),
    "fresh_sample_seed": sample_seed,
})
display(pd.DataFrame([
    {"Boundary kind": kind, "Program population": population_counts[kind], "Fresh sample": n}
    for kind, n in selection_counts.items()
]))


## Freeze the candidate and private review packets

Write only IDs, offsets, counts, dispositions and hashes to the local candidate tables. The private QA packets contain the complete parent text, every original chunk range and the exact-hold status of each chunk. Keep these packets and per-parent receipts out of the repository.

In [ ]:
output = ROOT / ".cache/chunk-policy-review/program-chunk-qa-v1"
tables = {
    "review_dispositions": review_ledger,
    "parent_holds": parent_holds,
    "exact_holds": exact_holds,
    "reference_candidate_offsets": model_candidate,
    "qa_sample": selection,
    "qa_strata": sample_strata,
}
if not output.exists():
    sample_chunk_rows = (
        candidate.join(F.broadcast(selection.select("release_row")), "release_row")
        .join(
            exact_holds.select(*KEY).withColumn("held_for_exact_reuse", F.lit(True)),
            KEY, "left",
        )
        .groupBy("release_row")
        .agg(F.sort_array(F.collect_list(F.struct(
            "ordinal", "start", "end", "reference_token_count", "chunk_sha256",
            F.coalesce("held_for_exact_reuse", F.lit(False)).alias("held_for_exact_reuse"),
        ))).alias("chunks"))
    )
    packet_source = (
        selection.join(
            parents.select("release_row", "clean_sha256", "clean_text"),
            ["release_row", "clean_sha256"],
        )
        .join(sample_chunk_rows, "release_row")
        .orderBy("draw_key", "release_row")
    )
    with tempfile.TemporaryDirectory(prefix=".program-qa-", dir=output.parent) as tmp:
        stage = Path(tmp) / "snapshot"
        stage.mkdir(mode=0o700)
        (stage / "packets").mkdir(mode=0o700)
        (stage / "decisions").mkdir(mode=0o700)
        file_index = {}
        for name, frame in tables.items():
            folder = stage / name
            frame.write.mode("errorifexists").option("compression", "zstd").parquet(str(folder))
            file_index[name] = [
                {
                    "path": path.relative_to(stage).as_posix(),
                    "sha256": sha256_file(path),
                    "rows": pq.ParquetFile(path).metadata.num_rows,
                }
                for path in sorted(folder.glob("*.parquet"))
            ]

        packet_size = 20
        batch_index = {}
        seen_parents = set()
        for idx, row in enumerate(packet_source.toLocalIterator()):
            if idx % packet_size == 0:
                batch = f"program-qa-{idx // packet_size + 1:03d}"
                batch_index[batch] = {"file": f"packets/{batch}.jsonl", "parents": 0}
            item = row.asDict(recursive=True)
            if row.release_row in seen_parents or [chunk.end for chunk in row.chunks] != row.ends:
                raise ValueError("A review packet does not match the selected parent geometry.")
            item["packet_sha256"] = canonical_sha256(item)
            with (stage / batch_index[batch]["file"]).open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(item, ensure_ascii=False) + "\n")
            seen_parents.add(row.release_row)
            batch_index[batch]["parents"] += 1
        if len(seen_parents) != sum(targets.values()):
            raise ValueError("The packet set does not cover the frozen QA draw.")
        for meta in batch_index.values():
            path = stage / meta["file"]
            path.chmod(0o400)
            meta["sha256"] = sha256_file(path)

        result = {
            **identity, "reference_candidate_audit": summary, "tables": file_index,
            "qa_batches": batch_index,
        }
        (stage / "MANIFEST.json").write_text(
            json.dumps(result, indent=2, sort_keys=True) + "\n", encoding="utf-8"
        )
        if sha256_file(closure_path) != identity["full_review_closure_sha256"]:
            raise ValueError("The input closure changed while creating QA packets.")
        if output.exists():
            raise FileExistsError("A QA snapshot was created concurrently.")
        stage.rename(output)

published = json.loads((output / "MANIFEST.json").read_text(encoding="utf-8"))
if (
    any(published.get(key) != value for key, value in identity.items())
    or published["reference_candidate_audit"] != summary
):
    raise ValueError("The existing snapshot belongs to a different policy or result.")
for name, parts in published["tables"].items():
    if not parts or {path.relative_to(output).as_posix()
                     for path in (output / name).glob("*.parquet")} != {
                         part["path"] for part in parts
                     }:
        raise ValueError(f"Unexpected snapshot file list for {name}.")
    for part in parts:
        path = output / part["path"]
        if sha256_file(path) != part["sha256"]:
            raise ValueError(f"A persisted {name} file changed.")
for meta in published["qa_batches"].values():
    if sha256_file(output / meta["file"]) != meta["sha256"]:
        raise ValueError("A frozen QA packet file changed.")

published_candidate = spark.read.parquet(str(output / "reference_candidate_offsets"))
verified_chunks = published_candidate.count()
if verified_chunks != summary["reference_candidate_chunks"]:
    raise ValueError("The persisted candidate row count did not reconcile.")
print(f"Verified a local reference candidate with {verified_chunks:,} chunks.")
print(f"Prepared {len(published['qa_batches'])} frozen semantic QA batches.")


## Verify semantic QA receipts

Review each whole parent in the frozen frame once. Inspect every internal boundary and use both sides to judge whether it separates a negation or qualification, an attributed claim or quotation, a question and answer, or a claim from evidence needed to understand it. For single-chunk controls, check that the visible chunk covers the complete text and record uncertainty if the evidence cannot be judged. Also check whether exact-held chunks remove evidence required by the retained chunks. Record this separately from boundary quality.

Each batch receipt in `decisions/<batch>.jsonl` has one decision per packet, with `release_row`, `packet_sha256`, `reviewer`, `boundary_outcome`, `affected_internal_ends`, `model_view_evidence_loss`, and a concise English `reason`. Boundary outcomes are `none`, `boundary_preference`, `material_evidence_break`, and `uncertain`. Record actual unresolved cases separately from boundary preferences. Do not extrapolate from an incomplete sample. A completed audit reports weighted results overall and by boundary kind and format. Any proposed offset change requires reference and selected-model budget checks and a fresh exact leakage check.

In [ ]:
allowed = {"none", "boundary_preference", "material_evidence_break", "uncertain"}
completed = []
for batch, meta in published["qa_batches"].items():
    receipt_path = output / "decisions" / f"{batch}.jsonl"
    if not receipt_path.exists():
        continue
    packets = [
        json.loads(line) for line in (output / meta["file"]).read_text(encoding="utf-8").splitlines()
    ]
    expected = {packet.pop("packet_sha256"): packet for packet in packets}
    if len(expected) != meta["parents"] or any(
        canonical_sha256(packet) != digest for digest, packet in expected.items()
    ):
        raise ValueError(f"The roster or packet body did not reconcile for {batch}.")
    decisions = [
        json.loads(line) for line in receipt_path.read_text(encoding="utf-8").splitlines()
    ]
    if len(decisions) != len(expected) or {row["packet_sha256"] for row in decisions} != set(
        expected
    ):
        raise ValueError(f"The semantic QA receipt did not match batch {batch}.")
    for decision in decisions:
        packet = expected[decision["packet_sha256"]]
        outcome = decision["boundary_outcome"]
        concerns = decision["affected_internal_ends"]
        if (
            decision["release_row"] != packet["release_row"]
            or outcome not in allowed
            or not isinstance(concerns, list)
            or any(type(end) is not int for end in concerns)
            or concerns != sorted(set(concerns))
            or not set(concerns).issubset(packet["ends"][:-1])
            or (outcome == "material_evidence_break" and not concerns)
            or (outcome == "none" and concerns)
            or type(decision["model_view_evidence_loss"]) is not bool
            or (
                decision["model_view_evidence_loss"]
                and not any(chunk["held_for_exact_reuse"] for chunk in packet["chunks"])
            )
            or any(
                not isinstance(decision.get(key), str) or not decision[key].strip()
                for key in ("reviewer", "reason")
            )
        ):
            raise ValueError(f"Invalid semantic QA evidence in {batch}.")
        completed.append((packet, decision))

frequency = Counter(decision["boundary_outcome"] for _, decision in completed)
print(f"Semantic QA receipts verified: {len(completed):,} of {sum(targets.values()):,} parents.")
print(f"Received findings: {dict(frequency)}")
if len(completed) == sum(targets.values()):
    completed_rows = [
        (
            packet["release_row"], packet["sampling_weight"], decision["boundary_outcome"],
            decision["model_view_evidence_loss"],
        )
        for packet, decision in completed
    ]
    outcomes = spark.createDataFrame(
        completed_rows, "release_row long, weight double, outcome string, evidence_loss boolean"
    ).join(selection.select("release_row", *strata), "release_row")
    for grouping in ([], ["boundary_kind"], ["format"]):
        display(outcomes.groupBy(*grouping).agg(
            F.count("*").alias("reviewed"),
            (
                F.sum(F.when(F.col("outcome") == "material_evidence_break", F.col("weight"))
                      .otherwise(0)) / F.sum("weight")
            ).alias("weighted_material_boundary_share"),
            (
                F.sum(F.when(F.col("outcome") == "uncertain", F.col("weight")).otherwise(0))
                / F.sum("weight")
            ).alias("weighted_uncertain_share"),
            (
                F.sum(F.when(F.col("evidence_loss"), F.col("weight")).otherwise(0))
                / F.sum("weight")
            ).alias("weighted_exact_hold_evidence_loss_share"),
        ).toPandas())
else:
    print("Weighted semantic rates require all frozen sample batches.")
spark.stop()
